In [1]:
import json
import time
from typing import Any, Dict, List, Optional

import pandas as pd
import requests


class PncpSearchPremiumClient:
    BASE_URL = "https://pncp.gov.br/api/search/"
    BASE_SITE_URL = "https://pncp.gov.br"

    def __init__(
        self,
        timeout: int = 60,
        sleep_seconds: float = 0.2,
        session: Optional[requests.Session] = None,
    ) -> None:
        self.timeout = timeout
        self.sleep_seconds = sleep_seconds
        self.session = session or requests.Session()
        self.session.headers.update(
            {
                "User-Agent": "Mozilla/5.0",
                "Accept": "application/json",
            }
        )

    def _get(self, params: Dict[str, Any]) -> Dict[str, Any]:
        response = self.session.get(
            self.BASE_URL,
            params=params,
            timeout=self.timeout,
        )

        print("[DEBUG URL]", response.url)

        if response.status_code == 400:
            return {
                "items": [],
                "total": 0,
                "_http_400": True,
            }

        response.raise_for_status()
        return response.json()

    def _build_url(self, item_url: Optional[str]) -> Optional[str]:
        if not item_url:
            return None
        if item_url.startswith("http://") or item_url.startswith("https://"):
            return item_url
        return f"{self.BASE_SITE_URL}{item_url}"

    def normalizar_item(self, item: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "objetoCompra": item.get("description"),
            "titulo": item.get("title"),
            "id": item.get("id"),
            "numeroControlePNCP": item.get("numero_controle_pncp"),
            "anoCompra": item.get("ano"),
            "sequencialCompra": item.get("numero_sequencial"),
            "numero": item.get("numero"),
            "documentType": item.get("document_type"),
            "tipoId": item.get("tipo_id"),
            "tipoNome": item.get("tipo_nome"),
            "modalidadeLicitacaoId": item.get("modalidade_licitacao_id"),
            "modalidadeLicitacaoNome": item.get("modalidade_licitacao_nome"),
            "situacaoId": item.get("situacao_id"),
            "situacaoNome": item.get("situacao_nome"),
            "orgaoId": item.get("orgao_id"),
            "orgaoCnpj": item.get("orgao_cnpj"),
            "orgaoNome": item.get("orgao_nome"),
            "unidadeId": item.get("unidade_id"),
            "unidadeCodigo": item.get("unidade_codigo"),
            "unidadeNome": item.get("unidade_nome"),
            "esferaId": item.get("esfera_id"),
            "esferaNome": item.get("esfera_nome"),
            "poderId": item.get("poder_id"),
            "poderNome": item.get("poder_nome"),
            "municipioId": item.get("municipio_id"),
            "municipioNome": item.get("municipio_nome"),
            "uf": item.get("uf"),
            "dataPublicacaoPncp": item.get("data_publicacao_pncp"),
            "dataAtualizacaoPncp": item.get("data_atualizacao_pncp"),
            "dataInicioVigencia": item.get("data_inicio_vigencia"),
            "dataFimVigencia": item.get("data_fim_vigencia"),
            "dataAssinatura": item.get("data_assinatura"),
            "cancelado": item.get("cancelado"),
            "temResultado": item.get("tem_resultado"),
            "valorGlobal": item.get("valor_global"),
            "fonteOrcamentaria": item.get("fonte_orcamentaria"),
            "fonteOrcamentariaId": item.get("fonte_orcamentaria_id"),
            "fonteOrcamentariaNome": item.get("fonte_orcamentaria_nome"),
            "itemUrl": item.get("item_url"),
            "urlPncp": self._build_url(item.get("item_url")),
            "createdAt": item.get("createdAt"),
            "raw": item,
        }

    def buscar_pagina(
        self,
        pagina: int = 1,
        tam_pagina: int = 100,
        tipos_documento: Optional[List[str]] = None,
        status: Optional[str] = None,
        ordenacao: str = "-data",
        params_extras: Optional[Dict[str, Any]] = None,
    ) -> Dict[str, Any]:
        params: Dict[str, Any] = {
            "pagina": pagina,
            "tam_pagina": tam_pagina,
            "ordenacao": ordenacao,
        }

        if status:
            params["status"] = status

        if tipos_documento:
            if len(tipos_documento) == 1:
                params["tipos_documento"] = tipos_documento[0]
            else:
                params["tipos_documento"] = ",".join(tipos_documento)

        if params_extras:
            params.update(params_extras)

        data = self._get(params)

        if data.get("_http_400"):
            return {
                "pagina": pagina,
                "tam_pagina": tam_pagina,
                "total": None,
                "quantidade_itens": 0,
                "items": [],
                "items_normalizados": [],
                "raw": data,
                "http_400": True,
            }

        items = data.get("items", [])
        total = data.get("total", 0)

        return {
            "pagina": pagina,
            "tam_pagina": tam_pagina,
            "total": total,
            "quantidade_itens": len(items),
            "items": items,
            "items_normalizados": [self.normalizar_item(x) for x in items],
            "raw": data,
            "http_400": False,
        }

    def buscar_todas_paginas(
        self,
        tam_pagina: int = 100,
        tipos_documento: Optional[List[str]] = None,
        status: Optional[str] = None,
        ordenacao: str = "-data",
        params_extras: Optional[Dict[str, Any]] = None,
        max_paginas: Optional[int] = None,
        deduplicar_por_id: bool = True,
        verbose: bool = True,
    ) -> Dict[str, Any]:
        pagina = 1
        total_esperado: Optional[int] = None
        itens_brutos: List[Dict[str, Any]] = []
        itens_normalizados: List[Dict[str, Any]] = []
        ids_vistos = set()

        while True:
            if max_paginas is not None and pagina > max_paginas:
                if verbose:
                    print(f"[INFO] Limite de páginas atingido: {max_paginas}")
                break

            resultado = self.buscar_pagina(
                pagina=pagina,
                tam_pagina=tam_pagina,
                tipos_documento=tipos_documento,
                status=status,
                ordenacao=ordenacao,
                params_extras=params_extras,
            )

            if resultado.get("http_400"):
                if verbose:
                    print(f"[INFO] API retornou 400 na página {pagina}. Encerrando paginação.")
                break

            items = resultado["items"]
            items_norm = resultado["items_normalizados"]
            total = resultado["total"]

            if total_esperado is None and total is not None:
                total_esperado = total

            if verbose:
                print(
                    f"[INFO] Página {pagina} | "
                    f"itens na página: {len(items)} | "
                    f"acumulado: {len(itens_normalizados)} | "
                    f"total API: {total_esperado}"
                )

            if not items:
                if verbose:
                    print("[INFO] Página vazia encontrada. Encerrando paginação.")
                break

            for bruto, norm in zip(items, items_norm):
                if deduplicar_por_id:
                    item_id = bruto.get("id")
                    if item_id and item_id in ids_vistos:
                        continue
                    if item_id:
                        ids_vistos.add(item_id)

                itens_brutos.append(bruto)
                itens_normalizados.append(norm)

            if len(items) < tam_pagina:
                if verbose:
                    print("[INFO] Última página detectada (menos itens que o tamanho da página).")
                break

            pagina += 1

            if self.sleep_seconds > 0:
                time.sleep(self.sleep_seconds)

        return {
            "total_informado_api": total_esperado,
            "total_coletado": len(itens_normalizados),
            "items": itens_normalizados,
            "items_brutos": itens_brutos,
        }

    def buscar_por_palavra_chave(
        self,
        termo: str,
        tam_pagina: int = 10,
        tipos_documento: Optional[List[str]] = None,
        status: Optional[str] = "recebendo_proposta",
        ordenacao: str = "-data",
        max_paginas: Optional[int] = None,
        deduplicar_por_id: bool = True,
        somente_colunas_principais: bool = True,
        verbose: bool = True,
    ) -> pd.DataFrame:
        """
        Busca por palavra-chave e retorna um DataFrame com os resultados.

        Parâmetros:
            termo: texto a buscar (ex: "controle de acesso")
            tam_pagina: número de itens por página (padrão: 10)
            tipos_documento: ex: ["edital"] (padrão: ["edital"])
            status: filtro de status (padrão: "recebendo_proposta")
            ordenacao: critério de ordenação (padrão: "-data")
            max_paginas: limite de páginas a buscar (None = sem limite)
            deduplicar_por_id: remove duplicatas pelo campo 'id'
            somente_colunas_principais: retorna apenas as colunas mais úteis
            verbose: exibe logs de progresso
        """
        if tipos_documento is None:
            tipos_documento = ["edital"]

        resultado = self.buscar_todas_paginas(
            tam_pagina=tam_pagina,
            tipos_documento=tipos_documento,
            status=status,
            ordenacao=ordenacao,
            params_extras={"q": termo},
            max_paginas=max_paginas,
            deduplicar_por_id=deduplicar_por_id,
            verbose=verbose,
        )

        items = resultado["items"]

        if not items:
            print(f"[INFO] Nenhum resultado encontrado para: '{termo}'")
            return pd.DataFrame()

        df = pd.DataFrame(items)

        if somente_colunas_principais:
            colunas = [
                "numeroControlePNCP",
                "objetoCompra",
                "orgaoNome",
                "municipioNome",
                "uf",
                "modalidadeLicitacaoNome",
                "situacaoNome",
                "dataPublicacaoPncp",
                "dataFimVigencia",
                "valorGlobal",
                "urlPncp",
            ]
            colunas_existentes = [c for c in colunas if c in df.columns]
            df = df[colunas_existentes]

        return df

    def salvar_json(
        self,
        resultado: Dict[str, Any],
        caminho_arquivo: str,
        somente_items: bool = False,
    ) -> None:
        conteudo = resultado["items"] if somente_items else resultado
        with open(caminho_arquivo, "w", encoding="utf-8") as f:
            json.dump(conteudo, f, ensure_ascii=False, indent=2)

    def salvar_csv(
        self,
        resultado: Dict[str, Any],
        caminho_arquivo: str,
        somente_colunas_principais: bool = False,
    ) -> None:
        df = pd.json_normalize(resultado["items"])

        if somente_colunas_principais:
            colunas = [
                "objetoCompra",
                "titulo",
                "numeroControlePNCP",
                "anoCompra",
                "sequencialCompra",
                "tipoNome",
                "modalidadeLicitacaoNome",
                "situacaoNome",
                "orgaoNome",
                "unidadeNome",
                "municipioNome",
                "uf",
                "dataPublicacaoPncp",
                "dataAtualizacaoPncp",
                "dataInicioVigencia",
                "dataFimVigencia",
                "urlPncp",
            ]
            colunas_existentes = [c for c in colunas if c in df.columns]
            df = df[colunas_existentes]

        df.to_csv(caminho_arquivo, index=False, encoding="utf-8-sig")


In [2]:
from datetime import date, datetime, timedelta

# Coleta recursiva por janelas de datas para contornar o limite da API
# A API aparenta bloquear na página 101 (≈100 páginas × tam_pagina itens -> ~10k).
client = PncpSearchPremiumClient(sleep_seconds=0.25)


def daterange_days(start: date, end: date):
    """Yield dates from start to end inclusive."""
    cur = start
    while cur <= end:
        yield cur
        cur += timedelta(days=1)


# resultado acumulado e set de ids para deduplicação
todos_itens = []
ids_vistos = set()

# caches e marcações para evitar reprocessamento / chamadas duplicadas
_processed_ranges = set()   # ranges já totalmente processados
_in_progress = set()        # ranges em processamento (evita reentrância)
_query_cache = {}           # cache simples por (start,end,tam_pagina,status,ordenacao)

MAX_ITEMS_PER_QUERY = 100 * 100  # tam_pagina(100) * max_pages(100) -> 10000


def collect_range(start: date, end: date, tam_pagina: int = 100, depth: int = 0, max_depth: int = 6):
    """Coleta itens no intervalo [start, end].

    Se a consulta retornar exatamente o limite (≈10k), divide o intervalo em duas
    metades e coleta recursivamente. Se chegar a um único dia e ainda assim
    retornar no limite, interrompe e registra um aviso (não há como contornar
    automaticamente sem filtros adicionais).

    Implementa proteção contra reprocessamento e cache de consultas idênticas
    para evitar chamadas duplicadas e URLs repetidos nos logs.
    """
    global todos_itens, ids_vistos, _processed_ranges, _in_progress, _query_cache

    if start > end:
        return

    range_key = (start.strftime("%Y%m%d"), end.strftime("%Y%m%d"), tam_pagina)

    # Se já processado, pule
    if range_key in _processed_ranges:
        print(f"[DEBUG] intervalo {start}→{end} já processado; pulando")
        return

    # Se já em processamento (reentrância), pule — isso evita duas chamadas idênticas
    if range_key in _in_progress:
        print(f"[DEBUG] intervalo {start}→{end} já em progresso; pulando")
        return

    _in_progress.add(range_key)
    try:
        # Chave de cache para evitar chamar buscar_todas_paginas muitas vezes com os mesmos params
        cache_key = (
            start.strftime("%Y%m%d"),
            end.strftime("%Y%m%d"),
            tam_pagina,
            "recebendo_proposta",
            "-data",
        )

        if cache_key in _query_cache:
            resultado = _query_cache[cache_key]
            print(f"[DEBUG] cache HIT para {start}→{end} (tam_pagina={tam_pagina})")
        else:
            params_extras = {"dataInicial": start.strftime("%Y%m%d"), "dataFinal": end.strftime("%Y%m%d")}
            resultado = client.buscar_todas_paginas(
                tam_pagina=tam_pagina,
                tipos_documento=["edital"],
                status="recebendo_proposta",
                ordenacao="-data",
                params_extras=params_extras,
                verbose=False,
            )
            _query_cache[cache_key] = resultado

        coletados = resultado.get("total_coletado") or 0
        total_api = resultado.get("total_informado_api")

        print(f"[DEBUG] intervalo {start} → {end} | API: {total_api} | coletados: {coletados} | depth={depth}")

        # Se a coleta não atingiu o limite, anexa os itens (deduplicando)
        if coletados < MAX_ITEMS_PER_QUERY:
            novos = 0
            for item in resultado.get("items", []):
                item_id = item.get("id")
                if item_id and item_id in ids_vistos:
                    continue
                if item_id:
                    ids_vistos.add(item_id)
                todos_itens.append(item)
                novos += 1
            print(f"  anexados: {novos} (acumulado: {len(todos_itens)})")
            # marca como processado com sucesso
            _processed_ranges.add(range_key)
            return

        # Aqui: coletados >= MAX_ITEMS_PER_QUERY (possível corte da API)
        # Se já está em profundidade máxima, não podemos subdividir mais com segurança
        if depth >= max_depth:
            print(f"[WARN] Intervalo {start}→{end} atingiu o limite e não foi possível subdividir mais (depth={depth}).\n" \
                  "Considere filtrar por outros campos (ex: orgao, municipio) ou pedir suporte da API.")
            # Ainda assim, tenta anexar os itens retornados (parcial)
            anexados = 0
            for item in resultado.get("items", []):
                item_id = item.get("id")
                if item_id and item_id in ids_vistos:
                    continue
                if item_id:
                    ids_vistos.add(item_id)
                todos_itens.append(item)
                anexados += 1
            print(f"  anexados (parciais): {anexados} (acumulado: {len(todos_itens)})")
            _processed_ranges.add(range_key)
            return

        # Se o intervalo tem mais de 1 dia, subdivida por dias (metade dos dias)
        delta_days = (end - start).days
        if delta_days >= 1:
            mid = start + timedelta(days=delta_days // 2)
            # subdivide: [start, mid] e [mid+1, end]
            collect_range(start, mid, tam_pagina=tam_pagina, depth=depth + 1, max_depth=max_depth)
            collect_range(mid + timedelta(days=1), end, tam_pagina=tam_pagina, depth=depth + 1, max_depth=max_depth)
            # após subdividir, marcamos a janela como processada
            _processed_ranges.add(range_key)
            return

        # Se aqui, start == end (um único dia) e mesmo assim excedeu o limite.
        # Não sabemos se API aceita granularidade menor (horas). Emitir aviso e anexar parcial.
        print(f"[WARN] Dia único {start} retornou >= {MAX_ITEMS_PER_QUERY} itens.\n" \
              "A API bloqueia páginas após a 100ª página; considere aplicar filtros adicionais (ex: orgao, municipio) ou contatar o provedor.")
        anexados = 0
        for item in resultado.get("items", []):
            item_id = item.get("id")
            if item_id and item_id in ids_vistos:
                continue
            if item_id:
                ids_vistos.add(item_id)
            todos_itens.append(item)
            anexados += 1
        print(f"  anexados (parciais): {anexados} (acumulado: {len(todos_itens)})")
        _processed_ranges.add(range_key)

    finally:
        # remove da lista de em progresso (se ainda estiver lá)
        _in_progress.discard(range_key)


# =======================
# Executar coleta para o intervalo desejado
# =======================
# Ajuste 'inicio' conforme pedido (ex: 2026-03-19)
inicio = date(2026, 3, 21)
fim = date.today()

print(f"Iniciando coleta: {inicio} → {fim}")
collect_range(inicio, fim)

print(f"\n✅ Total final: {len(todos_itens)} itens únicos")

resultado_final = {"total_coletado": len(todos_itens), "items": todos_itens}
client.salvar_json(resultado_final, "pncp_resultado.json")
client.salvar_csv(resultado_final, "pncp_resultado.csv")
print("✅ Arquivos salvos: pncp_resultado.json / pncp_resultado.csv")


Iniciando coleta: 2026-03-21 → 2026-03-21
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=1&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital&dataInicial=20260321&dataFinal=20260321
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=1&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital&dataInicial=20260321&dataFinal=20260321
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=2&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital&dataInicial=20260321&dataFinal=20260321
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=2&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital&dataInicial=20260321&dataFinal=20260321
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=3&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital&dataInicial=20260321&dataFinal=20260321
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=3&tam_pagina=100&ordenacao=-data&status=re

In [5]:
import json
from pathlib import Path


ARQUIVO_1 = Path("/Users/jose-cleiton/dev/script_pncp/pncp_contratacoes_premium.json")
ARQUIVO_2 = Path("/Users/jose-cleiton/dev/script_pncp/pncp_resultado.json")
ARQUIVO_SAIDA = Path("/Users/jose-cleiton/dev/script_pncp/pncp_merge_unicos.json")


def carregar_json(caminho: Path):
    with open(caminho, "r", encoding="utf-8") as f:
        return json.load(f)


def extrair_items(conteudo):
    """
    Suporta estes formatos:
    1) lista direta
    2) dict com chave 'items'
    3) dict com chave 'dados'
    4) dict premium com chave 'modalidades' -> cada modalidade -> 'dados'
    """
    if isinstance(conteudo, list):
        print(f"[INFO] JSON em formato lista direta: {len(conteudo)} itens")
        return conteudo

    if isinstance(conteudo, dict):
        if "items" in conteudo and isinstance(conteudo["items"], list):
            print(f"[INFO] JSON com chave 'items': {len(conteudo['items'])} itens")
            return conteudo["items"]

        if "dados" in conteudo and isinstance(conteudo["dados"], list):
            print(f"[INFO] JSON com chave 'dados': {len(conteudo['dados'])} itens")
            return conteudo["dados"]

        if "modalidades" in conteudo and isinstance(conteudo["modalidades"], dict):
            items = []
            for nome_modalidade, bloco in conteudo["modalidades"].items():
                if isinstance(bloco, dict) and isinstance(bloco.get("dados"), list):
                    qtd = len(bloco["dados"])
                    print(f"[INFO] Coletando modalidade '{nome_modalidade}': {qtd} itens")
                    items.extend(bloco["dados"])

            print(f"[INFO] Total extraído de 'modalidades': {len(items)} itens")
            return items

        print("[DEBUG] Chaves disponíveis no dict:", list(conteudo.keys()))
        raise ValueError("Não foi possível encontrar uma lista de itens válida no JSON.")

    raise ValueError(f"Formato inesperado no arquivo: {type(conteudo).__name__}")


def obter_numero_controle(item):
    """
    Tenta localizar o número de controle em vários formatos possíveis.
    """
    return (
        item.get("numeroControlePNCP")
        or item.get("numero_controle_pncp")
        or item.get("numero_controle_pncp".lower())
        or item.get("raw", {}).get("numero_controle_pncp")
        or item.get("raw", {}).get("numeroControlePNCP")
    )


def obter_data_atualizacao(item):
    """
    Tenta localizar a data de atualização em vários formatos possíveis.
    Retorna string ISO para comparação lexicográfica.
    """
    return (
        item.get("dataAtualizacaoPncp")
        or item.get("data_atualizacao_pncp")
        or item.get("datas", {}).get("atualizacao")
        or item.get("raw", {}).get("data_atualizacao_pncp")
        or item.get("raw", {}).get("dataAtualizacaoPncp")
        or ""
    )


def merge_unicos_por_numero_controle(items1, items2):
    unicos = {}
    sem_chave = []

    for item in items1 + items2:
        chave = obter_numero_controle(item)

        if not chave:
            sem_chave.append(item)
            continue

        if chave not in unicos:
            unicos[chave] = item
        else:
            atual = unicos[chave]
            if obter_data_atualizacao(item) > obter_data_atualizacao(atual):
                unicos[chave] = item

    resultado_items = list(unicos.values()) + sem_chave

    return {
        "total_arquivo_1": len(items1),
        "total_arquivo_2": len(items2),
        "total_unicos_por_numero_controle_pncp": len(unicos),
        "total_sem_numero_controle_pncp": len(sem_chave),
        "total_final": len(resultado_items),
        "items": resultado_items,
    }


def main():
    conteudo_1 = carregar_json(ARQUIVO_1)
    conteudo_2 = carregar_json(ARQUIVO_2)

    items_1 = extrair_items(conteudo_1)
    items_2 = extrair_items(conteudo_2)

    print(f"[INFO] Itens arquivo 1: {len(items_1)}")
    print(f"[INFO] Itens arquivo 2: {len(items_2)}")

    resultado_merge = merge_unicos_por_numero_controle(items_1, items_2)

    with open(ARQUIVO_SAIDA, "w", encoding="utf-8") as f:
        json.dump(resultado_merge, f, ensure_ascii=False, indent=2)

    print("\nMerge concluído com sucesso.")
    print(f"Total único por numero_controle_pncp: {resultado_merge['total_unicos_por_numero_controle_pncp']}")
    print(f"Total sem chave: {resultado_merge['total_sem_numero_controle_pncp']}")
    print(f"Total final: {resultado_merge['total_final']}")
    print(f"Saída: {ARQUIVO_SAIDA}")


if __name__ == "__main__":
    main()

[INFO] Coletando modalidade 'Concorrência - Eletrônica': 7 itens
[INFO] Coletando modalidade 'Concorrência - Presencial': 2 itens
[INFO] Coletando modalidade 'Pregão - Eletrônico': 13 itens
[INFO] Coletando modalidade 'Pregão - Presencial': 2 itens
[INFO] Coletando modalidade 'Dispensa': 66 itens
[INFO] Coletando modalidade 'Manifestação de Interesse': 1 itens
[INFO] Coletando modalidade 'Credenciamento': 14 itens
[INFO] Total extraído de 'modalidades': 105 itens
[INFO] JSON com chave 'items': 10000 itens
[INFO] Itens arquivo 1: 105
[INFO] Itens arquivo 2: 10000

Merge concluído com sucesso.
Total único por numero_controle_pncp: 10104
Total sem chave: 0
Total final: 10104
Saída: /Users/jose-cleiton/dev/script_pncp/pncp_merge_unicos.json

Merge concluído com sucesso.
Total único por numero_controle_pncp: 10104
Total sem chave: 0
Total final: 10104
Saída: /Users/jose-cleiton/dev/script_pncp/pncp_merge_unicos.json


In [7]:
import json
from copy import deepcopy
from pathlib import Path
from typing import Any, Dict, List, Optional


ARQUIVO_SEARCH = Path("/Users/jose-cleiton/dev/script_pncp/pncp_resultado.json")
ARQUIVO_CONSULTA = Path("/Users/jose-cleiton/dev/script_pncp/pncp_contratacoes_premium.json")
ARQUIVO_SAIDA = Path("/Users/jose-cleiton/dev/script_pncp/pncp_merge_normalizado.json")


def carregar_json(caminho: Path) -> Any:
    with open(caminho, "r", encoding="utf-8") as f:
        return json.load(f)


def extrair_items_search(conteudo: Any) -> List[Dict[str, Any]]:
    if isinstance(conteudo, dict) and isinstance(conteudo.get("items"), list):
        return conteudo["items"]
    if isinstance(conteudo, list):
        return conteudo
    raise ValueError("Não foi possível extrair items do JSON de search.")


def extrair_items_consulta(conteudo: Any) -> List[Dict[str, Any]]:
    # caso direto: {"data": [...]}
    if isinstance(conteudo, dict) and isinstance(conteudo.get("data"), list):
        return conteudo["data"]

    # caso premium: {"modalidades": {"Pregão": {"dados": [...]}, ...}}
    if isinstance(conteudo, dict) and isinstance(conteudo.get("modalidades"), dict):
        todos = []
        for nome_modalidade, bloco in conteudo["modalidades"].items():
            if isinstance(bloco, dict) and isinstance(bloco.get("dados"), list):
                print(f"[INFO] Coletando modalidade '{nome_modalidade}': {len(bloco['dados'])} itens")
                todos.extend(bloco["dados"])
        print(f"[INFO] Total extraído de 'modalidades': {len(todos)} itens")
        return todos

    # fallback: lista direta
    if isinstance(conteudo, list):
        return conteudo

    raise ValueError("Não foi possível extrair items do JSON de consulta.")


def obter_valor(d: Dict[str, Any], *caminhos, default=None):
    for caminho in caminhos:
        atual = d
        ok = True
        for chave in caminho:
            if not isinstance(atual, dict) or chave not in atual:
                ok = False
                break
            atual = atual[chave]
        if ok and atual is not None:
            return atual
    return default


def normalizar_item_search(item: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "numeroControlePNCP": item.get("numeroControlePNCP") or item.get("numero_controle_pncp"),
        "objetoCompra": item.get("objetoCompra") or item.get("description"),
        "titulo": item.get("titulo") or item.get("title"),
        "anoCompra": item.get("anoCompra") or item.get("ano"),
        "sequencialCompra": item.get("sequencialCompra") or item.get("numero_sequencial"),
        "numeroCompra": item.get("numero"),
        "processo": None,
        "modalidadeId": item.get("modalidadeLicitacaoId") or item.get("modalidade_licitacao_id"),
        "modalidadeNome": item.get("modalidadeLicitacaoNome") or item.get("modalidade_licitacao_nome"),
        "tipoInstrumentoConvocatorioCodigo": item.get("tipoId") or item.get("tipo_id"),
        "tipoInstrumentoConvocatorioNome": item.get("tipoNome") or item.get("tipo_nome"),
        "situacaoCompraId": item.get("situacaoId") or item.get("situacao_id"),
        "situacaoCompraNome": item.get("situacaoNome") or item.get("situacao_nome"),
        "orgaoEntidade": {
            "cnpj": item.get("orgaoCnpj") or item.get("orgao_cnpj"),
            "razaoSocial": item.get("orgaoNome") or item.get("orgao_nome"),
            "poderId": item.get("poderId") or item.get("poder_id"),
            "esferaId": item.get("esferaId") or item.get("esfera_id"),
        },
        "unidadeOrgao": {
            "codigoUnidade": item.get("unidadeCodigo") or item.get("unidade_codigo"),
            "nomeUnidade": item.get("unidadeNome") or item.get("unidade_nome"),
            "municipioNome": item.get("municipioNome") or item.get("municipio_nome"),
            "ufSigla": item.get("uf"),
        },
        "dataPublicacaoPncp": item.get("dataPublicacaoPncp") or item.get("data_publicacao_pncp"),
        "dataAtualizacaoGlobal": item.get("dataAtualizacaoPncp") or item.get("data_atualizacao_pncp"),
        "dataAberturaProposta": item.get("dataInicioVigencia") or item.get("data_inicio_vigencia"),
        "dataEncerramentoProposta": item.get("dataFimVigencia") or item.get("data_fim_vigencia"),
        "valorTotalEstimado": item.get("valorGlobal") or item.get("valor_global"),
        "valorTotalHomologado": None,
        "modoDisputaId": None,
        "modoDisputaNome": None,
        "linkSistemaOrigem": item.get("urlPncp") or item.get("itemUrl") or item.get("item_url"),
        "linkProcessoEletronico": None,
        "fontesOrcamentarias": item.get("fonteOrcamentaria") or item.get("fonte_orcamentaria") or [],
        "usuarioNome": None,
        "raw": item,
        "_origem": "search",
    }


def normalizar_item_consulta(item: Dict[str, Any]) -> Dict[str, Any]:
    # se já vier praticamente normalizado do premium
    numero = (
        item.get("numeroControlePNCP")
        or item.get("numero_controle_pncp")
        or obter_valor(item, ("raw", "numero_controle_pncp"))
    )

    if "orgaoEntidade" in item or "unidadeOrgao" in item or "dataAberturaProposta" in item:
        return {
            "numeroControlePNCP": item.get("numeroControlePNCP"),
            "objetoCompra": item.get("objetoCompra"),
            "titulo": item.get("tipoInstrumentoConvocatorioNome"),
            "anoCompra": item.get("anoCompra"),
            "sequencialCompra": item.get("sequencialCompra"),
            "numeroCompra": item.get("numeroCompra"),
            "processo": item.get("processo"),
            "modalidadeId": item.get("modalidadeId"),
            "modalidadeNome": item.get("modalidadeNome"),
            "tipoInstrumentoConvocatorioCodigo": item.get("tipoInstrumentoConvocatorioCodigo"),
            "tipoInstrumentoConvocatorioNome": item.get("tipoInstrumentoConvocatorioNome"),
            "situacaoCompraId": item.get("situacaoCompraId"),
            "situacaoCompraNome": item.get("situacaoCompraNome"),
            "orgaoEntidade": item.get("orgaoEntidade") or {},
            "unidadeOrgao": item.get("unidadeOrgao") or {},
            "dataPublicacaoPncp": item.get("dataPublicacaoPncp"),
            "dataAtualizacaoGlobal": item.get("dataAtualizacaoGlobal") or item.get("dataAtualizacao"),
            "dataAberturaProposta": item.get("dataAberturaProposta"),
            "dataEncerramentoProposta": item.get("dataEncerramentoProposta"),
            "valorTotalEstimado": item.get("valorTotalEstimado"),
            "valorTotalHomologado": item.get("valorTotalHomologado"),
            "modoDisputaId": item.get("modoDisputaId"),
            "modoDisputaNome": item.get("modoDisputaNome"),
            "linkSistemaOrigem": item.get("linkSistemaOrigem"),
            "linkProcessoEletronico": item.get("linkProcessoEletronico"),
            "fontesOrcamentarias": item.get("fontesOrcamentarias") or [],
            "usuarioNome": item.get("usuarioNome"),
            "raw": item,
            "_origem": "consulta",
        }

    # se vier no formato normalizado do premium antigo
    return {
        "numeroControlePNCP": numero,
        "objetoCompra": item.get("objeto_compra"),
        "titulo": item.get("tipo_instrumento_convocatorio_nome"),
        "anoCompra": item.get("ano_compra"),
        "sequencialCompra": item.get("sequencial_compra"),
        "numeroCompra": item.get("numero_compra"),
        "processo": item.get("processo"),
        "modalidadeId": item.get("modalidade_id"),
        "modalidadeNome": item.get("modalidade_nome"),
        "tipoInstrumentoConvocatorioCodigo": item.get("tipo_instrumento_convocatorio_codigo"),
        "tipoInstrumentoConvocatorioNome": item.get("tipo_instrumento_convocatorio_nome"),
        "situacaoCompraId": item.get("situacao_compra_id"),
        "situacaoCompraNome": item.get("situacao_compra_nome"),
        "orgaoEntidade": {
            "cnpj": obter_valor(item, ("orgao_entidade", "cnpj")),
            "razaoSocial": obter_valor(item, ("orgao_entidade", "razao_social")),
            "poderId": obter_valor(item, ("orgao_entidade", "poder_id")),
            "esferaId": obter_valor(item, ("orgao_entidade", "esfera_id")),
        },
        "unidadeOrgao": {
            "codigoUnidade": obter_valor(item, ("unidade_orgao", "codigo_unidade")),
            "nomeUnidade": obter_valor(item, ("unidade_orgao", "nome_unidade")),
            "municipioNome": obter_valor(item, ("unidade_orgao", "municipio_nome")),
            "ufSigla": obter_valor(item, ("unidade_orgao", "uf_sigla")),
        },
        "dataPublicacaoPncp": obter_valor(item, ("datas", "publicacao_pncp")),
        "dataAtualizacaoGlobal": (
            obter_valor(item, ("datas", "atualizacao_global"))
            or obter_valor(item, ("datas", "atualizacao"))
        ),
        "dataAberturaProposta": obter_valor(item, ("datas", "abertura_proposta")),
        "dataEncerramentoProposta": obter_valor(item, ("datas", "encerramento_proposta")),
        "valorTotalEstimado": item.get("valor_total_estimado"),
        "valorTotalHomologado": item.get("valor_total_homologado"),
        "modoDisputaId": item.get("modo_disputa_id"),
        "modoDisputaNome": item.get("modo_disputa_nome"),
        "linkSistemaOrigem": obter_valor(item, ("links", "sistema_origem")),
        "linkProcessoEletronico": obter_valor(item, ("links", "processo_eletronico")),
        "fontesOrcamentarias": item.get("fontes_orcamentarias") or [],
        "usuarioNome": item.get("usuario_nome"),
        "raw": item,
        "_origem": "consulta",
    }


def data_referencia(item: Dict[str, Any]) -> str:
    return (
        item.get("dataAtualizacaoGlobal")
        or item.get("dataPublicacaoPncp")
        or item.get("dataEncerramentoProposta")
        or ""
    )


def preferir_item(item_atual: Dict[str, Any], novo_item: Dict[str, Any]) -> Dict[str, Any]:
    # mantém o mais recente
    if data_referencia(novo_item) > data_referencia(item_atual):
        base = deepcopy(novo_item)
    else:
        base = deepcopy(item_atual)

    origens = {item_atual.get("_origem"), novo_item.get("_origem")}
    if origens == {"search", "consulta"}:
        base["_origem"] = "ambos"

    # tenta preencher campos vazios do escolhido com o outro
    outro = novo_item if base is item_atual else item_atual

    for chave, valor in outro.items():
        if chave not in base or base[chave] in (None, "", [], {}):
            base[chave] = valor

    return base


def merge_por_numero_controle(
    itens_search: List[Dict[str, Any]],
    itens_consulta: List[Dict[str, Any]],
) -> Dict[str, Any]:
    mapa: Dict[str, Dict[str, Any]] = {}
    sem_chave: List[Dict[str, Any]] = []

    for item in itens_search:
        norm = normalizar_item_search(item)
        chave = norm.get("numeroControlePNCP")
        if not chave:
            sem_chave.append(norm)
            continue
        if chave in mapa:
            mapa[chave] = preferir_item(mapa[chave], norm)
        else:
            mapa[chave] = norm

    for item in itens_consulta:
        norm = normalizar_item_consulta(item)
        chave = norm.get("numeroControlePNCP")
        if not chave:
            sem_chave.append(norm)
            continue
        if chave in mapa:
            mapa[chave] = preferir_item(mapa[chave], norm)
        else:
            mapa[chave] = norm

    items_finais = list(mapa.values()) + sem_chave

    resumo_origem = {"search": 0, "consulta": 0, "ambos": 0}
    for item in mapa.values():
        origem = item.get("_origem")
        if origem in resumo_origem:
            resumo_origem[origem] += 1

    return {
        "resumo": {
            "total_search": len(itens_search),
            "total_consulta": len(itens_consulta),
            "total_unicos_com_chave": len(mapa),
            "total_sem_chave": len(sem_chave),
            "total_final": len(items_finais),
            "por_origem": resumo_origem,
        },
        "items": sorted(
            items_finais,
            key=lambda x: (
                x.get("dataEncerramentoProposta") or "",
                x.get("dataPublicacaoPncp") or "",
                x.get("numeroControlePNCP") or "",
            )
        ),
    }


def main():
    conteudo_search = carregar_json(ARQUIVO_SEARCH)
    conteudo_consulta = carregar_json(ARQUIVO_CONSULTA)

    itens_search = extrair_items_search(conteudo_search)
    itens_consulta = extrair_items_consulta(conteudo_consulta)

    print(f"[INFO] JSON search: {len(itens_search)} itens")
    print(f"[INFO] JSON consulta: {len(itens_consulta)} itens")

    resultado = merge_por_numero_controle(itens_search, itens_consulta)

    with open(ARQUIVO_SAIDA, "w", encoding="utf-8") as f:
        json.dump(resultado, f, ensure_ascii=False, indent=2)

    print("\nMerge concluído com sucesso.")
    print(json.dumps(resultado["resumo"], ensure_ascii=False, indent=2))
    print(f"\nArquivo salvo em: {ARQUIVO_SAIDA}")


if __name__ == "__main__":
    main()

[INFO] Coletando modalidade 'Concorrência - Eletrônica': 7 itens
[INFO] Coletando modalidade 'Concorrência - Presencial': 2 itens
[INFO] Coletando modalidade 'Pregão - Eletrônico': 13 itens
[INFO] Coletando modalidade 'Pregão - Presencial': 2 itens
[INFO] Coletando modalidade 'Dispensa': 66 itens
[INFO] Coletando modalidade 'Manifestação de Interesse': 1 itens
[INFO] Coletando modalidade 'Credenciamento': 14 itens
[INFO] Total extraído de 'modalidades': 105 itens
[INFO] JSON search: 9974 itens
[INFO] JSON consulta: 105 itens

Merge concluído com sucesso.
{
  "total_search": 9974,
  "total_consulta": 105,
  "total_unicos_com_chave": 10037,
  "total_sem_chave": 0,
  "total_final": 10037,
  "por_origem": {
    "search": 9932,
    "consulta": 63,
    "ambos": 42
  }
}

Arquivo salvo em: /Users/jose-cleiton/dev/script_pncp/pncp_merge_normalizado.json


In [3]:
from datetime import date, timedelta

client = PncpSearchPremiumClient(sleep_seconds=0.3)

# ── Estratégia: dividir por janelas de datas ──────────────────────────────────
# A API limita 100 páginas × 100 itens = 10.000 por consulta.
# Quebrando por semana, cada fatia fica bem abaixo do limite.

def gerar_janelas(data_inicio: date, data_fim: date, dias: int = 7):
    """Gera pares (inicio, fim) com intervalos de `dias` dias."""
    atual = data_inicio
    while atual < data_fim:
        fim_janela = min(atual + timedelta(days=dias - 1), data_fim)
        yield atual, fim_janela
        atual = fim_janela + timedelta(days=1)

todos_itens = []
ids_vistos = set()

inicio = date(2025, 1, 1)
fim    = date.today()

for janela_inicio, janela_fim in gerar_janelas(inicio, fim, dias=7):
    params_extras = {
        "dataInicial": janela_inicio.strftime("%Y%m%d"),
        "dataFinal":   janela_fim.strftime("%Y%m%d"),
    }

    resultado = client.buscar_todas_paginas(
        tam_pagina=100,
        tipos_documento=["edital"],
        status="recebendo_proposta",
        ordenacao="-data",
        params_extras=params_extras,
        verbose=False,          # True para ver cada página
    )

    novos = 0
    for item in resultado["items"]:
        item_id = item.get("id")
        if item_id and item_id in ids_vistos:
            continue
        if item_id:
            ids_vistos.add(item_id)
        todos_itens.append(item)
        novos += 1

    print(
        f"[{janela_inicio} → {janela_fim}] "
        f"coletados: {resultado['total_coletado']} | "
        f"novos únicos: {novos} | "
        f"total acumulado: {len(todos_itens)}"
    )

print(f"\n✅ Total final: {len(todos_itens)} itens únicos")

# Salvar resultado completo
resultado_final = {"total_coletado": len(todos_itens), "items": todos_itens}
client.salvar_json(resultado_final, "pncp_resultado.json")
client.salvar_csv(resultado_final,  "pncp_resultado.csv")


[DEBUG URL] https://pncp.gov.br/api/search/?pagina=1&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital&dataInicial=20250101&dataFinal=20250107
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=2&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital&dataInicial=20250101&dataFinal=20250107
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=2&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital&dataInicial=20250101&dataFinal=20250107
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=3&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital&dataInicial=20250101&dataFinal=20250107
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=3&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital&dataInicial=20250101&dataFinal=20250107
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=4&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital&da

KeyboardInterrupt: 